# 10 OpenOximetry Canonical Window Export

Build canonical 10s/5s window features for passlisted OpenOximetry encounters with timing-offset corrections and encounter-level quality gates.

Inputs from: `outputs/openox_reference_encounter_passlist.csv` (from notebook 09) and `outputs/openox_ppg_eligible_encounters.csv` (from notebook 06).
Outputs to: `outputs/openox_canonical_window_features.csv`, `outputs/openox_canonical_label_summary.csv`, `outputs/openox_canonical_recording_summary.csv`, and `outputs/openox_canonical_export_audit.csv`.
Next: notebook 11 for grouped logistic-regression baseline on the canonical OpenOximetry feature set.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal, stats


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "outputs" / "openox_reference_encounter_passlist.csv").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing outputs/openox_reference_encounter_passlist.csv and notebooks/")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PASSLIST_CSV = OUTPUTS_DIR / "openox_reference_encounter_passlist.csv"
ELIGIBLE_CSV = OUTPUTS_DIR / "openox_ppg_eligible_encounters.csv"
FEATURE_CSV = OUTPUTS_DIR / "openox_canonical_window_features.csv"
LABEL_SUMMARY_CSV = OUTPUTS_DIR / "openox_canonical_label_summary.csv"
RECORDING_SUMMARY_CSV = OUTPUTS_DIR / "openox_canonical_recording_summary.csv"

TARGET_RATE_HZ = 50.0
LOW_CUT_HZ = 0.5
HIGH_CUT_HZ = 8.0
FILTER_ORDER = 4
WINDOW_SECONDS = 10.0
STRIDE_SECONDS = 5.0
LOW_SPO2_THRESHOLD = 90.0
BORDERLINE_MARGIN = 1.0

passlist_df = pd.read_csv(PASSLIST_CSV)
eligible_df = pd.read_csv(ELIGIBLE_CSV)
work_df = eligible_df.merge(passlist_df[["encounter_id", "reference_column"]], on="encounter_id", how="inner")

print(f"Passlist encounters: {len(passlist_df)}")
print(f"Canonical work encounters: {len(work_df)}")

In [ ]:
def choose_ppg_channel(channel_labels: list[str]) -> int:
    lowered = [str(label).strip().lower() for label in channel_labels]
    for preferred in ("ir signal", "ir", "red signal", "red"):
        for idx, label in enumerate(lowered):
            if preferred == label:
                return idx
    raise ValueError(f"Could not choose a signal channel from labels: {channel_labels}")


def read_ppg_frames(dat_path: Path, channel_count: int) -> np.ndarray:
    raw = np.memmap(dat_path, dtype=np.int32, mode="r")
    return np.asarray(raw).reshape(-1, channel_count)


def resample_and_filter(values: np.ndarray, source_rate_hz: float) -> np.ndarray:
    target_length = int(round(len(values) * TARGET_RATE_HZ / source_rate_hz))
    resampled = signal.resample(values.astype(np.float64), target_length)
    sos = signal.butter(
        FILTER_ORDER,
        [LOW_CUT_HZ, HIGH_CUT_HZ],
        btype="bandpass",
        fs=TARGET_RATE_HZ,
        output="sos",
    )
    return signal.sosfiltfilt(sos, resampled)

## Phase 1: Time-Offset Audit

Quantify signed and absolute timing offsets between PPG and 2Hz references to verify alignment assumptions before feature extraction.

In [ ]:
# Load timing information from eligible encounters
timing_cols = ["encounter_id", "ppg_start_time_seconds", "hz2_first_time_seconds", "abs_time_delta_seconds", "timing_eligible_le_10s"]
timing_df = eligible_df[timing_cols].copy()

# Compute signed delta: positive means PPG started before 2Hz first sample
timing_df["signed_delta_seconds"] = timing_df["ppg_start_time_seconds"] - timing_df["hz2_first_time_seconds"]

# Summary statistics
print("=== TIME-OFFSET AUDIT ===")
print(f"\nTiming offset statistics (signed delta in seconds):")
print(f"  Mean: {timing_df['signed_delta_seconds'].mean():.4f} s")
print(f"  Std:  {timing_df['signed_delta_seconds'].std():.4f} s")
print(f"  Min:  {timing_df['signed_delta_seconds'].min():.4f} s")
print(f"  Max:  {timing_df['signed_delta_seconds'].max():.4f} s")
print(f"\nAbsolute timing offset (before signing):")
print(f"  Mean: {timing_df['abs_time_delta_seconds'].mean():.4f} s")
print(f"  Max:  {timing_df['abs_time_delta_seconds'].max():.4f} s")
print(f"\nEncounters with timing offset > 2s: {(timing_df['abs_time_delta_seconds'] > 2.0).sum()}")
print(f"Encounters with timing offset > 5s: {(timing_df['abs_time_delta_seconds'] > 5.0).sum()}")

## Phase 2: Windowing Setup And Per-Window Quality Rules

Define minimum reference-sample requirements, apply signed timing offsets per encounter, and prepare gate criteria used during canonical window generation.

In [ ]:
# Define thresholds for filtering
MIN_REFERENCE_SAMPLES_PER_WINDOW = 10  # At 2 Hz, 10s window should have ~20 points; 10 is conservative min
MIN_WINDOWS_PER_ENCOUNTER = 5  # Encounters must produce at least this many non-dropped windows

# Create a mapping of encounter_id to signed_delta for quick lookup during windowing
delta_map = dict(zip(timing_df["encounter_id"], timing_df["signed_delta_seconds"]))

# Track window and encounter statistics for audit
encounter_audit = {
    "encounter_id": [],
    "recording_id": [],
    "initial_windows_proposed": [],
    "passed_per_window_checks": [],
    "passed_per_encounter_check": [],
    "final_kept_windows": [],
    "skip_reason": [],
}

print("=== WINDOWING SETUP ===")
print(f"Minimum reference samples per window: {MIN_REFERENCE_SAMPLES_PER_WINDOW}")
print(f"Minimum windows per encounter: {MIN_WINDOWS_PER_ENCOUNTER}")

In [ ]:
def compute_window_features(values: np.ndarray, sample_rate_hz: float) -> dict[str, float]:
    mean_value = float(np.mean(values))
    std_value = float(np.std(values))
    minimum_value = float(np.min(values))
    maximum_value = float(np.max(values))
    value_range = maximum_value - minimum_value
    rms_value = float(np.sqrt(np.mean(np.square(values))))

    skewness_value = float(stats.skew(values, bias=False))
    if not np.isfinite(skewness_value):
        skewness_value = 0.0

    kurtosis_value = float(stats.kurtosis(values, fisher=True, bias=False))
    if not np.isfinite(kurtosis_value):
        kurtosis_value = 0.0

    energy_value = float(np.sum(np.square(values)))
    derivative_std_value = float(np.std(np.diff(values))) if len(values) > 1 else 0.0

    frequencies, power = signal.welch(values, fs=sample_rate_hz, nperseg=min(256, len(values)))
    if len(frequencies) <= 1 or len(power) <= 1:
        dominant_frequency_hz = 0.0
        bandpower_0_5_2_hz = 0.0
        bandpower_2_5_hz = 0.0
    else:
        positive_frequencies = frequencies[1:]
        positive_power = power[1:]
        dominant_frequency_hz = float(positive_frequencies[int(np.argmax(positive_power))]) if len(positive_power) else 0.0

        mask_0_5_2 = (frequencies >= 0.5) & (frequencies <= 2.0)
        mask_2_5 = (frequencies >= 2.0) & (frequencies <= 5.0)
        bandpower_0_5_2_hz = float(np.trapezoid(power[mask_0_5_2], frequencies[mask_0_5_2])) if np.any(mask_0_5_2) else 0.0
        bandpower_2_5_hz = float(np.trapezoid(power[mask_2_5], frequencies[mask_2_5])) if np.any(mask_2_5) else 0.0

    bandpower_ratio = bandpower_0_5_2_hz / bandpower_2_5_hz if bandpower_2_5_hz > 0.0 else 0.0

    return {
        "mean": mean_value,
        "std": std_value,
        "minimum": minimum_value,
        "maximum": maximum_value,
        "value_range": value_range,
        "rms": rms_value,
        "skewness": skewness_value,
        "kurtosis": kurtosis_value,
        "energy": energy_value,
        "derivative_std": derivative_std_value,
        "dominant_frequency_hz": float(dominant_frequency_hz),
        "bandpower_0_5_2_hz": float(bandpower_0_5_2_hz),
        "bandpower_2_5_hz": float(bandpower_2_5_hz),
        "bandpower_ratio_0_5_2_to_2_5": float(bandpower_ratio),
    }

In [ ]:
# Track windows per encounter (for per-encounter gating)
windows_per_encounter = {}
feature_rows = []
window_size_samples = int(round(WINDOW_SECONDS * TARGET_RATE_HZ))
stride_size_samples = int(round(STRIDE_SECONDS * TARGET_RATE_HZ))
borderline_lower = LOW_SPO2_THRESHOLD - BORDERLINE_MARGIN
borderline_upper = LOW_SPO2_THRESHOLD + BORDERLINE_MARGIN

for row in work_df.itertuples(index=False):
    dat_path = Path(str(row.ppg_hea_path)).with_suffix(".dat")
    hea_path = Path(str(row.ppg_hea_path))
    hz2_path = Path(str(row.hz2_csv_path))

    with hea_path.open("r", encoding="utf-8", errors="ignore") as handle:
        lines = [line.strip() for line in handle if line.strip()]
    first_tokens = lines[0].split()
    channel_count = int(first_tokens[1])
    source_rate_hz = float(first_tokens[2])
    channel_labels = [" ".join(line.split()[8:]) if len(line.split()) >= 9 else "" for line in lines[1 : 1 + channel_count]]

    channel_index = choose_ppg_channel(channel_labels)
    frames = read_ppg_frames(dat_path, channel_count)
    filtered_values = resample_and_filter(frames[:, channel_index], source_rate_hz)

    ref_df = pd.read_csv(hz2_path)
    ref_df["Timestamp"] = pd.to_datetime(ref_df["Timestamp"], errors="coerce")
    ref_df[row.reference_column] = pd.to_numeric(ref_df[row.reference_column], errors="coerce")
    ref_df = ref_df.loc[ref_df["Timestamp"].notna() & ref_df[row.reference_column].notna()].copy()
    if ref_df.empty:
        continue

    t0 = ref_df["Timestamp"].iloc[0]
    ref_df["elapsed_seconds"] = (ref_df["Timestamp"] - t0).dt.total_seconds()

    for start_index in range(0, len(filtered_values) - window_size_samples + 1, stride_size_samples):
        end_index = start_index + window_size_samples
        start_time_seconds = start_index / TARGET_RATE_HZ
        end_time_seconds = end_index / TARGET_RATE_HZ

        # FIX #1: Apply per-encounter signed time delta to align PPG time with 2Hz reference time
        signed_delta = delta_map.get(row.encounter_id, 0.0)
        start_time_adjusted = start_time_seconds + signed_delta
        end_time_adjusted = end_time_seconds + signed_delta

        window_ref = ref_df.loc[
            (ref_df["elapsed_seconds"] >= start_time_adjusted)
            & (ref_df["elapsed_seconds"] < end_time_adjusted),
            row.reference_column,
        ]
        if window_ref.empty:
            continue

        # FIX #2: Enforce minimum per-window samples (at 2 Hz, 10s window should have ~20 samples; 10 is conservative threshold)
        if len(window_ref) < MIN_REFERENCE_SAMPLES_PER_WINDOW:
            continue

        median_spo2 = float(window_ref.median())
        min_spo2 = float(window_ref.min())
        is_borderline = borderline_lower <= median_spo2 <= borderline_upper
        if is_borderline:
            continue

        values = filtered_values[start_index:end_index]
        feature_rows.append(
            {
                "recording_id": str(row.encounter_id),
                "start_time_seconds": float(start_time_seconds),
                "end_time_seconds": float(end_time_seconds),
                "sample_rate_hz": float(TARGET_RATE_HZ),
                "median_spo2": float(median_spo2),
                "min_spo2": float(min_spo2),
                "label": int(median_spo2 < LOW_SPO2_THRESHOLD),
                "label_min_spo2": int(min_spo2 < LOW_SPO2_THRESHOLD),
                **compute_window_features(values, TARGET_RATE_HZ),
            }
        )

print("=")

## Phase 3: Per-Encounter Gate And Artifact Export

Apply encounter-level minimum-window criteria, export the canonical feature table, and write audit/summary artifacts for baseline modeling in notebook 11.

In [ ]:
# FIX #3: Group windows by encounter and apply per-encounter minimum-window gate
# Build a temporary DF to group and filter
temp_df = pd.DataFrame(feature_rows)

# Count windows per encounter
windows_by_encounter = temp_df.groupby("recording_id").size()

# Identify encounters passing the per-encounter gate
encounters_passing_gate = windows_by_encounter[windows_by_encounter >= MIN_WINDOWS_PER_ENCOUNTER].index.tolist()

# Filter features to only include encounters meeting the gate
feature_df_gated = temp_df[temp_df["recording_id"].isin(encounters_passing_gate)].copy()

# Build audit table for all 173 timing-eligible encounters
for enc_id in timing_df["encounter_id"]:
    enc_windows_before_gate = len(temp_df[temp_df["recording_id"] == str(enc_id)])
    enc_windows_after_gate = len(feature_df_gated[feature_df_gated["recording_id"] == str(enc_id)])
    
    # Determine skip reason
    if enc_windows_before_gate == 0:
        skip_reason = "no_windows_generated"
    elif enc_windows_after_gate == 0 and enc_windows_before_gate > 0:
        skip_reason = "per_encounter_windows_below_threshold"
    else:
        skip_reason = "passed"
    
    encounter_audit["encounter_id"].append(enc_id)
    encounter_audit["recording_id"].append(str(enc_id))
    encounter_audit["initial_windows_proposed"].append(enc_windows_before_gate)
    encounter_audit["passed_per_window_checks"].append(enc_windows_before_gate)  # Already filtered in loop
    encounter_audit["passed_per_encounter_check"].append(1 if enc_windows_after_gate > 0 else 0)
    encounter_audit["final_kept_windows"].append(enc_windows_after_gate)
    encounter_audit["skip_reason"].append(skip_reason)

# Create audit dataframe and export
audit_df = pd.DataFrame(encounter_audit)
AUDIT_CSV = OUTPUTS_DIR / "openox_canonical_export_audit.csv"
audit_df.to_csv(AUDIT_CSV, index=False)

# Sort gated features by recording_id and time
feature_df = feature_df_gated.sort_values(["recording_id", "start_time_seconds"]).reset_index(drop=True)

# Export corrected feature table
feature_df.to_csv(FEATURE_CSV, index=False)

print("=== PHASE 3 RESULTS ===")
print(f"Total windows before per-encounter gate: {len(temp_df)}")
print(f"Total windows after per-encounter gate: {len(feature_df)}")
print(f"Encounters surviving full pipeline: {len(encounters_passing_gate)} / {len(timing_df)}")
print(f"Windows dropped by per-encounter gate: {len(temp_df) - len(feature_df)}")
print(f"\nAudit table exported to: {AUDIT_CSV}")
print(f"Feature table exported to: {FEATURE_CSV}")

In [ ]:
label_summary_df = (
    feature_df.groupby("label", dropna=False)
    .agg(
        row_count=("recording_id", "size"),
        recording_count=("recording_id", "nunique"),
        mean_median_spo2=("median_spo2", "mean"),
    )
    .reset_index()
)
label_summary_df.to_csv(LABEL_SUMMARY_CSV, index=False)

recording_summary_df = (
    feature_df.groupby("recording_id", dropna=False)
    .agg(
        window_count=("label", "size"),
        positive_window_count=("label", "sum"),
        min_median_spo2=("median_spo2", "min"),
        max_median_spo2=("median_spo2", "max"),
    )
    .reset_index()
)
recording_summary_df["has_positive_window"] = recording_summary_df["positive_window_count"] > 0
recording_summary_df.to_csv(RECORDING_SUMMARY_CSV, index=False)

print(f"Feature rows: {len(feature_df)}")
print(f"Positive recordings: {int(recording_summary_df['has_positive_window'].sum())} / {len(recording_summary_df)}")